# 05 Models
For each combination of antibiotic, split and fold, the pipeline trains two gene-based models (logistic regression and gradient boosting) and a lineage-only baseline. Their performance is evaluated using balanced accuracy and AUROC.


In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, roc_auc_score
from sklearn.preprocessing import OneHotEncoder
import sys

src_dir = Path("../src").resolve()
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

import splits
import data
ANTIBIOTICS = data.ANTIBIOTICS
X, labels, year, clusters = data.load_data()

main_antibiotics = ["CIP", "GEN", "CXM"]

## Scoring

AUROC uses predicted probabilities; balanced accuracy uses yes/no predictions at a threshold of 0.5.

Both are 0.5 for a model that guesses, and 1.0 for a perfect model.

In [4]:
def score_model(model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    balanced_acc = balanced_accuracy_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_proba)
    return balanced_acc, roc_auc

## Gene models

A new copy of each model is trained on every fold. Both models use the same folds, so their scores can be compared. Settings are fixed and the same for every split.

In [6]:
from sklearn.base import clone
from sklearn.ensemble import HistGradientBoostingClassifier

rows = []

models = {
    "logistic": LogisticRegression(class_weight="balanced", max_iter=1000),
    "boosting": HistGradientBoostingClassifier(class_weight="balanced", random_state=42),
}

for antibiotic in main_antibiotics:
    all_splits = {
        "random": splits.random_split(labels, antibiotic),
        "lineage": splits.lineage_split(labels, clusters, antibiotic),
        "time": splits.time_split(labels, year, antibiotic),
    }
    for split_name, folds in all_splits.items():
        for i, (train_ids, test_ids) in enumerate(folds):
            X_train = X.loc[train_ids]
            X_test = X.loc[test_ids]
            y_train = (labels.loc[train_ids, antibiotic] == "R").astype(int)
            y_test = (labels.loc[test_ids, antibiotic] == "R").astype(int)

            for model_name, model in models.items():
                print(f"{antibiotic} | {split_name} | fold {i + 1} | {model_name}")
                fresh_model = clone(model)
                fresh_model.fit(X_train, y_train)
                bal_acc, auroc = score_model(fresh_model, X_test, y_test)
                rows.append({
                    "antibiotic": antibiotic,
                    "split": split_name,
                    "model": model_name,
                    "fold": i + 1,
                    "balanced_accuracy": bal_acc,
                    "AUROC": auroc,
                })

results = pd.DataFrame(rows)
Path("../outputs").mkdir(exist_ok=True)
results.to_csv("../outputs/results_gene_models.csv", index=False)

CIP | random | fold 1 | logistic
CIP | random | fold 1 | boosting


C:\Users\julia\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "C:\Users\julia\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "C:\Users\julia\anaconda3\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\julia\anaconda3\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "C:\Users\julia\anaconda3\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _winapi.CreatePro

CIP | random | fold 2 | logistic
CIP | random | fold 2 | boosting
CIP | random | fold 3 | logistic
CIP | random | fold 3 | boosting
CIP | random | fold 4 | logistic
CIP | random | fold 4 | boosting
CIP | random | fold 5 | logistic
CIP | random | fold 5 | boosting
CIP | lineage | fold 1 | logistic
CIP | lineage | fold 1 | boosting
CIP | lineage | fold 2 | logistic
CIP | lineage | fold 2 | boosting
CIP | lineage | fold 3 | logistic
CIP | lineage | fold 3 | boosting
CIP | lineage | fold 4 | logistic
CIP | lineage | fold 4 | boosting
CIP | lineage | fold 5 | logistic
CIP | lineage | fold 5 | boosting
CIP | time | fold 1 | logistic
CIP | time | fold 1 | boosting
GEN | random | fold 1 | logistic
GEN | random | fold 1 | boosting
GEN | random | fold 2 | logistic
GEN | random | fold 2 | boosting
GEN | random | fold 3 | logistic
GEN | random | fold 3 | boosting
GEN | random | fold 4 | logistic
GEN | random | fold 4 | boosting
GEN | random | fold 5 | logistic
GEN | random | fold 5 | boosting
GEN 

In [7]:
print(len(results))
results.groupby(["antibiotic", "split", "model"]).size()

66


antibiotic  split    model   
CIP         lineage  boosting    5
                     logistic    5
            random   boosting    5
                     logistic    5
            time     boosting    1
                     logistic    1
CXM         lineage  boosting    5
                     logistic    5
            random   boosting    5
                     logistic    5
            time     boosting    1
                     logistic    1
GEN         lineage  boosting    5
                     logistic    5
            random   boosting    5
                     logistic    5
            time     boosting    1
                     logistic    1
dtype: int64

66 results: 3 antibiotics × 11 folds (5 random, 5 lineage, 1 time) × 2 models.

In [9]:
results.groupby(["antibiotic", "split", "model"])[["balanced_accuracy", "AUROC"]].mean().round(3)

balanced_accuracy  AUROC
antibiotic split   model                             
CIP        lineage boosting              0.749  0.908
                   logistic              0.757  0.874
           random  boosting              0.901  0.967
                   logistic              0.911  0.960
           time    boosting              0.898  0.955
                   logistic              0.895  0.944
CXM        lineage boosting              0.814  0.863
                   logistic              0.776  0.842
           random  boosting              0.842  0.890
                   logistic              0.826  0.887
           time    boosting              0.732  0.814
                   logistic              0.741  0.814
GEN        lineage boosting              0.908  0.966
                   logistic              0.853  0.938
           random  boosting              0.922  0.965
                   logistic              0.893  0.958
           time    boosting              0.969  0.991
                   logistic              0.900  0.964

### Interpretation

Drop from the random to the lineage split:
- **GEN** (resistance from genes): small (AUROC about 0.00–0.02)
- **CXM** (mixed): medium (AUROC about 0.03–0.05)
- **CIP** (resistance from mutations): large (AUROC about 0.06–0.09, balanced accuracy about 0.15)

That matches the hypothesis.

## Lineage-only baseline

A logistic regression whose only input is the family label (cutoff 1168), with no genes. It shows how much can be predicted from family alone.

In [12]:
baseline_rows = []

for antibiotic in main_antibiotics:
    all_splits = {
        "random": splits.random_split(labels, antibiotic),
        "lineage": splits.lineage_split(labels, clusters, antibiotic),
        "time": splits.time_split(labels, year, antibiotic),
    }
    for split_name, folds in all_splits.items():
        for i, (train_ids, test_ids) in enumerate(folds):
            families_train = clusters.loc[train_ids, [splits.LINEAGE_COLUMN]]
            families_test = clusters.loc[test_ids, [splits.LINEAGE_COLUMN]]

            encoder = OneHotEncoder(handle_unknown="ignore")
            X_train = encoder.fit_transform(families_train)
            X_test = encoder.transform(families_test)

            y_train = (labels.loc[train_ids, antibiotic] == "R").astype(int)
            y_test = (labels.loc[test_ids, antibiotic] == "R").astype(int)

            model = LogisticRegression(class_weight="balanced", max_iter=1000)
            model.fit(X_train, y_train)
            bal_acc, auroc = score_model(model, X_test, y_test)

            baseline_rows.append({
                "antibiotic": antibiotic,
                "split": split_name,
                "model": "lineage_only",
                "fold": i + 1,
                "balanced_accuracy": bal_acc,
                "AUROC": auroc,
            })

baseline_results = pd.DataFrame(baseline_rows)
all_results = pd.concat([results, baseline_results], ignore_index=True)
all_results.to_csv("../outputs/results_all.csv", index=False)

In [13]:
print(len(all_results))
baseline_results.groupby(["antibiotic", "split"])[["balanced_accuracy", "AUROC"]].mean().round(3)

99


balanced_accuracy  AUROC
antibiotic split                            
CIP        lineage              0.500  0.500
           random               0.847  0.931
           time                 0.845  0.922
CXM        lineage              0.500  0.500
           random               0.693  0.793
           time                 0.677  0.753
GEN        lineage              0.500  0.500
           random               0.680  0.794
           time                 0.653  0.760

### Interpretation

- **Lineage split:** 0.5 for every antibiotic, as expected. All test families are new, so knowing families gives no information.
- **Random split:** family alone reaches AUROC 0.93 for CIP, almost as high as the gene models (0.96–0.97). For GEN it reaches 0.79, much lower than the gene models (0.96). So CIP's random-split score comes mostly from family, while GEN's comes from genes.
- **Time split:** family alone still reaches 0.92 for CIP, because the same families appear in both periods. The time split does not remove family information the way the lineage split does.

## Summary

Two results support the hypothesis:
1. Under the lineage split, CIP drops much more than GEN, CXM is in between.
2. Family alone explains almost all of CIP's random-split score, but much less of GEN's.

Results are saved in `outputs/results_all.csv` (99 rows).